# Example Data

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bobleesj/quantem.widget/blob/main/docs/tutorials/download_data.ipynb)

Dataset page: [bobleesj/quantem-data](https://huggingface.co/datasets/bobleesj/quantem-data)

Electron microscopists often want to try a viewer before finding, copying, or converting their own raw files. `quantem.data` provides a small public set of real microscopy datasets with calibration attached, so a notebook can say "load the gold HAADF example" and get a `Dataset2d` object ready for `Show2D`.

This page uses a real HAADF image of gold nanoparticles. The original array is 4096 by 4096 `uint16` data; the tutorial makes a calibrated preview crop so the documentation page opens quickly while the full cached file remains available for local work.

```{tip}
Run this exact notebook with the Colab badge above, or [View or download this notebook on GitHub](https://github.com/bobleesj/quantem.widget/blob/main/docs/tutorials/download_data.ipynb). For finished results, use [Saving and sharing](widget_export) to export interactive HTML or share a trusted notebook with widget state.
```


In [1]:
import numpy as np
import quantem.data as qdata

from quantem.core.datastructures import Dataset2d, Dataset3d
from quantem.widget import Show2D, Show3D

## Choose a microscopy dataset

Use `qdata.datasets()` when you want a readable catalog: dataset name, modality, size, and the exact load call. It does not download the large data files. `qdata.browse()` is also available when you want thumbnail cards in a notebook.


In [2]:
qdata.datasets(modality="haadf")

Name,Data,Size,Load
gold_drift_0deg,haadf,8.7 MB,"qdata.load(""gold_drift_0deg"")"
gold_drift_90deg,haadf,8.7 MB,"qdata.load(""gold_drift_90deg"")"
gold_haadf,haadf,33.8 MB,"qdata.load(""gold_haadf"")"
gold_haadf_npy,haadf,33.6 MB,"qdata.load(""gold_haadf_npy"")"


## Load a calibrated HAADF image

Load by the short dataset name. `quantem.data` downloads the file once into the local Hugging Face cache, reads its calibration sidecar, and returns a `Dataset2d`. Future runs reuse the local cache.


In [3]:
full_dataset = qdata.load("gold_haadf_npy", verbose=False)
full_image = full_dataset.array
pixel_size_nm = float(full_dataset.sampling[0])

print(f"Dataset: {full_dataset.name}")
print(f"Shape:   {full_image.shape[0]} x {full_image.shape[1]} pixels")
print(f"Counts:  {full_image.dtype}")
print(f"Pixel:   {pixel_size_nm:.4f} {full_dataset.units[0]}/pixel")

Dataset: gold_haadf_npy
Shape:   4096 x 4096 pixels
Counts:  uint16
Pixel:   0.0186 nm/pixel


## Show2D preview

For docs and Colab startup speed, make a 512 by 512 preview from the full HAADF image. The preview keeps the calibrated pixel size, so the scale bar remains physically meaningful.


In [4]:
stride = 8
preview = np.asarray(full_image[::stride, ::stride], dtype=np.float32)
preview -= preview.min()
preview /= preview.max() or 1.0

preview_dataset = Dataset2d.from_array(
    preview,
    sampling=(pixel_size_nm * stride, pixel_size_nm * stride),
    units=tuple(full_dataset.units),
    name="Gold HAADF preview",
)

Show2D(preview_dataset, cmap="inferno")

Show2D(512×512, cmap=inferno)

## Show3D from the same data

The same loaded image can seed a small stack for `Show3D`. This keeps the example lightweight while still using real HAADF texture and calibration.


In [5]:
shifts = np.linspace(-12, 12, 24).astype(int)
stack = np.stack([np.roll(preview, shift, axis=0) for shift in shifts]).astype(np.float32)
stack_dataset = Dataset3d.from_array(
    stack,
    sampling=(1.0, pixel_size_nm * stride, pixel_size_nm * stride),
    units=("frame", *tuple(full_dataset.units)),
    name="Gold HAADF preview stack",
)

Show3D(stack_dataset, fps=12, cmap="inferno")

Show3D(24×512×512, frame=12, cmap=inferno)

## Larger files

The same API works for larger 4D-STEM and EDS datasets. Choose those intentionally because they can be hundreds of MB or several GB.

```python
qdata.datasets()                     # readable names, modality, and size
path = qdata.download("gold_512")    # raw cached folder, no widget load yet
# data = qdata.load("gold_512", det_bin=4)  # load when you want the full 4D-STEM data
```

For shareable tutorials, embed a small calibrated preview and link to the full dataset page. For local analysis, load the full cached dataset and pass it directly into the widget.


## Widget tutorial shortcuts

The widget tutorials use `quantem.widget.datasets` when the goal is to open a ready-to-view example without thinking about Hugging Face folder names, strides, or monitor-file layout:

```python
from quantem.widget.datasets import show1d_ducky, show2d_gold, show3d_gold, show4dstem_gold
```

These helpers use the shared size language `small`, `medium`, `large`, and `full`. The Show1D ducky monitor is stored under `widget-tutorials/show1d/ducky/small` in the public Hugging Face dataset, so widget tutorial payloads stay grouped under one folder instead of polluting the dataset root.
